In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .getOrCreate()

# 1. Baca data dari HDFS dan tambahkan kolom pendapatan
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv"
df_transaksi = spark.read.csv(path_hdfs, header=True, inferSchema=True)

df_transaksi = df_transaksi.withColumn(
    "pendapatan", 
    col("unit_terjual") * col("harga_satuan")
)

# 2. Buat df_target dari dictionary data_target_cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Menampilkan hasil
df_transaksi.show(5)
df_target.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      400

In [5]:
from pyspark.sql.functions import sum as spark_sum, col

# 1. Meringkas total pendapatan per kota dari df_transaksi
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 2. Join dengan df_target berdasarkan kolom "kota"
df_bagian_a = ringkasan_kota.join(df_target, on="kota", how="inner")

# 3. Menambahkan kolom pencapaian_persen
df_bagian_a = df_bagian_a.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100
)

# 4. Mengurutkan hasil dari pencapaian tertinggi
df_bagian_a.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as spark_sum, col, row_number

# 1. Agregasi total pendapatan per kota dan kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 2. Definisikan window spesifikasi: partisi per kota, urutkan pendapatan terbesar
window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# 3. Terapkan row_number() dan filter untuk mengambil peringkat 1 saja (top-1)
df_bagian_b = df_kategori_kota.withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# 4. Tampilkan hasil
df_bagian_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [7]:
# 1. Pendaftaran DataFrame sebagai temporary view
df_transaksi.createOrReplaceTempView("v_transaksi")
df_target.createOrReplaceTempView("v_target")

# 2. Penulisan dan eksekusi kueri SQL
sql_query = """
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM v_transaksi t
    JOIN v_target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
"""

df_bagian_c = spark.sql(sql_query)

# 3. Tampilkan hasil
df_bagian_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Berdasarkan hasil pengolahan data pada Bagian A dan Bagian B, cabang yang berkinerja paling baik adalah **Purworejo** (PIC: Fitri). Cabang ini mencatatkan pencapaian target tertinggi sebesar **152,17%** dengan total pendapatan mencapai **Rp45.650.000** dari target bulanan sebesar **Rp30.000.000**. Kinerja positif cabang tersebut didorong utama oleh kategori **Kesehatan & Kecantikan** yang menjadi kontributor pendapatan terbesar mencapai **Rp10.075.000**.

Sebaliknya, cabang yang paling memerlukan perhatian manajemen adalah **Semarang** (PIC: Sari). Cabang ini mencatatkan persentase pencapaian target terendah, yaitu hanya sebesar **69,41%** dengan akumulasi pendapatan **Rp38.175.000** dari target bulanan yang cukup tinggi sebesar **Rp55.000.000**. Meskipun kategori **Rumah Tangga** menjadi produk unggulan di cabang ini dengan kontribusi **Rp11.125.000**, secara keseluruhan total omzet belum mampu mendekati target. Manajemen perlu melakukan evaluasi strategi pemasaran dan penyesuaian alokasi inventaris pada cabang Semarang.